# 011 — Sélection Random Forest
Validation croisée groupée par jardin, test indépendant. Accord avec règles synthétiques, pas vérité horticole.

In [ ]:
from pathlib import Path
import sys,json,hashlib,time
import numpy as np
import pandas as pd
import joblib
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_extraction import DictVectorizer
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import GroupShuffleSplit,GroupKFold,GridSearchCV
from sklearn.metrics import f1_score,make_scorer,confusion_matrix,classification_report
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p/"gdai-plant").is_dir())
PLANT = ROOT/"gdai-plant"
sys.path.insert(0,str(PLANT))
from api.rf_features import features
PAIRS = PLANT/"data/matching/fr-idf-v1/labelled-pairs.json"
raw = PAIRS.read_bytes()
rows = [r for r in json.loads(raw) if r.get("experimental_training_eligible") is True and r.get("plant_quality")=="ACCEPTED" and r.get("label_source")=="RULE_DERIVED" and r.get("relevance_grade") in (0,1,2,3)]
assert len({r["garden_id"] for r in rows}) >= 10
assert len({r["scientific_name"] for r in rows}) >= 5
assert len({(r["garden_id"],r["scientific_name"]) for r in rows}) == len(rows)
x = [features(r) for r in rows]
y = np.array([r["relevance_grade"] for r in rows])
groups = np.array([r["garden_id"] for r in rows])
tr,te = next(GroupShuffleSplit(n_splits=1,test_size=.25,random_state=42).split(x,y,groups))
assert not set(groups[tr]) & set(groups[te])
assert len(set(y[tr])) >= 2 and len(set(y[te])) >= 2
print(len(tr),len(te),pd.Series(y).value_counts())

In [ ]:
# 8 configurations x 3 folds, two CPU workers, no GPU required.
pipeline = make_pipeline(DictVectorizer(),RandomForestClassifier(random_state=42,class_weight="balanced",n_jobs=1))
grid = {"randomforestclassifier__n_estimators":[80,160],"randomforestclassifier__max_depth":[8,None],"randomforestclassifier__min_samples_leaf":[2,4]}
search = GridSearchCV(pipeline,grid,cv=GroupKFold(3),scoring=make_scorer(f1_score,labels=[0,1,2,3],average="macro",zero_division=0),n_jobs=2,error_score="raise")
start=time.monotonic()
search.fit([x[i] for i in tr],y[tr],groups=groups[tr])
elapsed=time.monotonic()-start
display(pd.DataFrame(search.cv_results_)[["params","mean_test_score","std_test_score","mean_fit_time"]].sort_values("mean_test_score",ascending=False))
print(search.best_params_,elapsed)

In [ ]:
model=search.best_estimator_
pred=model.predict([x[i] for i in te])
scores=model.predict_proba([x[i] for i in te])[:,model.classes_>=2].sum(axis=1)
precision, counts = [], []
for garden in sorted(set(groups[te])):
    pos=[j for j,i in enumerate(te) if groups[i]==garden and pred[j]>=2]
    top=sorted(pos,key=lambda j:(-scores[j],rows[te[j]]["scientific_name"]))[:5]
    counts.append(len(top))
    if top:
        precision.append(sum(y[te[j]]>=2 for j in top)/len(top))
metrics={"status":"EXPERIMENTAL_RULE_AGREEMENT_NOT_EXPERT_VALIDATION","input_sha256":hashlib.sha256(raw).hexdigest(),"best_params":search.best_params_,"cv_macro_f1":float(search.best_score_),"macro_f1":float(f1_score(y[te],pred,labels=[0,1,2,3],average="macro",zero_division=0)),"confusion_matrix":confusion_matrix(y[te],pred,labels=[0,1,2,3]).tolist(),"classification_report":classification_report(y[te],pred,labels=[0,1,2,3],output_dict=True,zero_division=0),"train_gardens":len(set(groups[tr])),"test_gardens":len(set(groups[te])),"distinct_eligible_plants":len({r["scientific_name"] for r in rows}),"mean_returned_precision":float(np.mean(precision)) if precision else None,"gardens_with_five":sum(c==5 for c in counts),"gardens_without_results":sum(c==0 for c in counts),"elapsed_seconds":elapsed}
display(pd.DataFrame(metrics["confusion_matrix"],index=range(4),columns=range(4)))
print(json.dumps(metrics,indent=2))

In [ ]:
OUT=PLANT/"models/rf-fr-idf-run01"
OUT.mkdir(parents=True,exist_ok=False)
joblib.dump(model,OUT/"rf.joblib")
(OUT/"metrics.json").write_text(json.dumps(metrics,indent=2),encoding="utf-8")
print(OUT)